# EgyptGPT Autoresearch

Autonomous AI research loop using Claude Code on your Claude subscription.
Based on Karpathy's [autoresearch](https://github.com/karpathy/autoresearch) pattern.

**Requirements:** Colab GPU runtime, Claude Pro/Max subscription.

## 1. Clone repo and install dependencies

In [ ]:
%cd /content
!rm -rf EgyptGPT
!git clone --recurse-submodules https://github.com/JLansey/EgyptGPT.git
%cd /content/EgyptGPT
!bash setup.sh

## 2. Mount Google Drive and prepare data

In [ ]:
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

%cd /content/EgyptGPT

DATA_DIR = 'data/egypt_char'
DRIVE_DATA = '/content/drive/MyDrive/EgyptGPT_autoresearch/data'
DATA_FILES = ['train.bin', 'val.bin', 'meta.pkl']

# Try loading cached data from Drive
cached = all(os.path.exists(os.path.join(DRIVE_DATA, f)) for f in DATA_FILES)
if cached:
    print('Loading cached data from Google Drive...')
    for f in DATA_FILES:
        shutil.copy(os.path.join(DRIVE_DATA, f), os.path.join(DATA_DIR, f))
else:
    print('No cache. Running prepare.py...')
    # Must run as module import so hiero_transformer submodule is on the path
    !cd /content/EgyptGPT && python -c "from data.egypt_char import prepare"
    # Cache to Drive
    os.makedirs(DRIVE_DATA, exist_ok=True)
    for f in DATA_FILES:
        shutil.copy(os.path.join(DATA_DIR, f), os.path.join(DRIVE_DATA, f))
    print(f'Cached to {DRIVE_DATA}')

# Verify — fail hard if anything is missing
for f in DATA_FILES:
    path = os.path.join(DATA_DIR, f)
    assert os.path.exists(path), f'MISSING: {path}'
    print(f'{path}: {os.path.getsize(path):,} bytes')
print('Data ready.')

## 3. Install Claude Code

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
!sudo apt-get install -y nodejs tmux
!npm install -g @anthropic-ai/claude-code
!claude --version

## 4. Create non-root user and configure everything

`--dangerously-skip-permissions` is blocked under root (Colab default).
This creates a non-root user with full ownership of the repo, git config, Drive backup dir,
and pre-creates the autoresearch branch.

In [ ]:
import subprocess

# Create non-root user
!useradd -m researcher 2>/dev/null || true

# Give researcher full ownership of the repo (including .git internals)
!chown -R researcher:researcher /content/EgyptGPT

# GPU access: open nvidia devices to all users
!chmod a+rw /dev/nvidia* /dev/dri/* 2>/dev/null || true

# Drive access: allow non-root users to access the FUSE mount
# (remount with allow_other; requires /etc/fuse.conf to permit it)
!sed -i 's/#user_allow_other/user_allow_other/' /etc/fuse.conf 2>/dev/null || true
!mount -o remount,allow_other /content/drive 2>/dev/null || fusermount -u /content/drive 2>/dev/null || true

# Re-mount drive if remount failed (will use allow_other now)
import os
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

# Give researcher access to Drive backup dir
!mkdir -p /content/drive/MyDrive/EgyptGPT_autoresearch
!chmod -R a+rw /content/drive/MyDrive/EgyptGPT_autoresearch

# Configure git for researcher
for cmd in [
    'git config --global --add safe.directory /content/EgyptGPT',
    'git config --global user.email autoresearch@colab',
    'git config --global user.name autoresearch',
]:
    subprocess.run(['su', '-c', cmd, 'researcher'])

# Pre-create the autoresearch branch
subprocess.run(['su', '-c', 'cd /content/EgyptGPT && git checkout -b autoresearch/run1', 'researcher'])

# Set Claude Code to use Opus model for researcher
!mkdir -p /home/researcher/.claude
!echo '{"model": "claude-opus-4-6"}' > /home/researcher/.claude/settings.json
!chown -R researcher:researcher /home/researcher/.claude

print('Non-root user ready, git configured, branch created, GPU + Drive accessible.')

## 5. Log in to Claude (one-time, uses your subscription)

Open the **Colab terminal** (bottom-left terminal icon) and run:

```bash
cd /EgyptGPT && claude
```

It will show an OAuth URL. **Copy the URL, paste it in your browser, and log in.**
Once authenticated, type `/exit` to quit, then run the next cell.

In [ ]:
# Copy Claude credentials from root to the researcher user
!cp -r /root/.claude /home/researcher/.claude 2>/dev/null || true
# Re-apply Opus model setting (credential copy may overwrite settings)
!python3 -c "
import json, os
p = '/home/researcher/.claude/settings.json'
s = json.load(open(p)) if os.path.exists(p) else {}
s['model'] = 'claude-opus-4-6'
json.dump(s, open(p,'w'))
"
!chown -R researcher:researcher /home/researcher/.claude
print('Credentials copied, Opus model confirmed.')

## 6. Launch autoresearch

Open the **Colab terminal** and run:

```bash
tmux new-session -s autoresearch
su - researcher
cd /content/EgyptGPT
claude --model claude-opus-4-6 --dangerously-skip-permissions
```

then paste in this prompt:
```
Read program.md in this directory. This is an autoresearch setup.
You are on branch autoresearch/run1. Data files are verified.

1. Initialize results.tsv with the header row
2. Begin the experiment loop as described in program.md

After each experiment, copy results.tsv to /content/drive/MyDrive/EgyptGPT_autoresearch/results.tsv so the log survives session crashes.

Start with the baseline run, then iterate autonomously. Never stop.
```

**Detach:** `Ctrl+B` then `D` (keeps it running in background)

**Reattach:** `tmux attach -t autoresearch`

**Stop:** `tmux kill-session -t autoresearch`

## 7. Monitor

In [ ]:
!cat /content/EgyptGPT/results.tsv 2>/dev/null || echo 'No results yet.'

In [ ]:
!cd /content/EgyptGPT && git log --oneline -20

In [ ]:
!tmux has-session -t autoresearch 2>/dev/null && echo 'Running' || echo 'Stopped'